In [3]:
from pyspark.sql import SparkSession

# ============================
#     CONFIGURAÇÃO DO SPARK
# ============================

DELTA_VERSION = "2.4.0"
DELTA_PACKAGE = f"io.delta:delta-core_2.12:{DELTA_VERSION}"

warehouse_location = "hdfs://hdfs-nn:9000/demo/silver"
db_name = "demo"
table_name = "titles_delta"
table_location = f"{warehouse_location}/{table_name}" # Define o caminho completo da tabela

spark = (
    SparkSession.builder
        .appName("View Silver - titles_delta")
        .master("local[*]")
        .config("spark.sql.warehouse.dir", warehouse_location)
        .config("hive.metastore.uris", "thrift://hive-metastore:9083")
        .config("spark.jars.packages", DELTA_PACKAGE)
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .enableHiveSupport()
        .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("\n✅ Spark iniciado e Delta/Hive carregados\n")

# ===============================================
#     LER E REGISTRAR A TABELA SILVER (Passo 1-3)
# ===============================================

# 1. Garante que o banco de dados (schema) exista
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}") 

# 2. Seleciona o DB
spark.sql(f"USE {db_name}")
print(f"📂 Base selecionada: {db_name}\n")

# 3. Registra a Tabela Delta no Metastore se ainda não estiver lá
print(f"🔗 Tentando registrar a tabela no HDFS: {table_location}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA 
    LOCATION '{table_location}'
""")
print(f"✅ Tabela {table_name} registrada com sucesso (ou já existia).\n")

# ===============================================
#     CONFIRMAÇÃO DA CAMADA SILVER (Passo 4)
# ===============================================

print("🕵️ CONFIRMAÇÃO DA LOCALIZAÇÃO E FORMATO (PROVA SILVER):")

# Verifica a Localização (deve apontar para /silver/)
print("  - Localização (Location):")
spark.sql(f"DESCRIBE EXTENDED {db_name}.{table_name}") \
    .filter("col_name = 'Location'") \
    .show(truncate=False)

# Verifica o Formato (deve ser 'delta')
print("  - Formato (Provider):")
spark.sql(f"DESCRIBE EXTENDED {db_name}.{table_name}") \
    .filter("col_name = 'Provider'") \
    .show(truncate=False)
    
print("---")

# ===============================================
#     ANÁLISES ORIGINAIS (Passo 5)
# ===============================================

# Mostrar schema e informações da tabela
print("📘 DESCRIBE EXTENDED titles_delta:")
spark.sql(f"DESCRIBE EXTENDED {table_name}").show(truncate=False)

# Mostrar linhas do Silver
print("\n📊 PRIMEIRAS 20 LINHAS DO SILVER titles_delta:")
spark.sql(f"SELECT * FROM {table_name}").show(20, truncate=False)

# Resumo por ano
print("\n📈 RESUMO POR ANO (release_year):")
spark.sql(f"""
    SELECT 
        release_year,
        COUNT(*) AS total_titles,
        ROUND(AVG(imdb_score),2) AS avg_score,
        MAX(imdb_score) AS max_score
    FROM {table_name}
    WHERE imdb_score IS NOT NULL
    GROUP BY release_year
    ORDER BY release_year DESC
""").show(truncate=False)


✅ Spark iniciado e Delta/Hive carregados

📂 Base selecionada: demo

🔗 Tentando registrar a tabela no HDFS: hdfs://hdfs-nn:9000/demo/silver/titles_delta
✅ Tabela titles_delta registrada com sucesso (ou já existia).

🕵️ CONFIRMAÇÃO DA LOCALIZAÇÃO E FORMATO (PROVA SILVER):
  - Localização (Location):
+--------+--------------------------------------------+-------+
|col_name|data_type                                   |comment|
+--------+--------------------------------------------+-------+
|Location|hdfs://hdfs-nn:9000/demo/silver/titles_delta|       |
+--------+--------------------------------------------+-------+

  - Formato (Provider):
+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
|Provider|delta    |       |
+--------+---------+-------+

---
📘 DESCRIBE EXTENDED titles_delta:
+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|id               |string   |null   |
|title            |

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# CONFIGURAÇÃO DE SPARK JÁ INCLUÍDA NO SEU AMBIENTE
db_name = "demo"
table_name = "titles_delta" # Ou titles_parquet

# Seleciona o DataFrame
titles_df = spark.table(f"{db_name}.{table_name}")

# Lista todas as colunas
cols = titles_df.columns
total_rows = titles_df.count()

print(f"\n🔬 Análise de Nulos e Vazios na Tabela {table_name}:")
print(f"Total de Linhas: {total_rows}")

# Cria uma expressão de agregação para contar nulos e strings vazias em cada coluna
summary_exprs = []
for col_name in cols:
    # Contar valores NULOS
    summary_exprs.append(
        F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias(f"NULL_Count_{col_name}")
    )
    # Contar STRINGS VAZIAS ('') - Apenas para colunas String
    if titles_df.schema[col_name].dataType.simpleString() == 'string':
         summary_exprs.append(
            F.sum(F.when(F.col(col_name) == '', 1).otherwise(0)).alias(f"EMPTY_Count_{col_name}")
        )

# Executa a agregação e transpõe o resultado para um formato legível
data_quality_df = titles_df.agg(*summary_exprs)

# Transpor o DataFrame para visualização
print("\n")
data_quality_df.show(truncate=False)

# NOTA: Se o seu titles_delta não tiver sido corrigido, você verá 'NULL_Count_imdb_score' com um valor alto
# e 'NULL_Count_genres' como 0, mas 'EMPTY_Count_genres' também como 0 (pois a transformação limpa-os ou preenche-os com 'Unknown').


🔬 Análise de Nulos e Vazios na Tabela titles_delta:
Total de Linhas: 5850


+-------------+--------------+----------------+-----------------+---------------+----------------+----------------------+-----------------------+-----------------------+----------------------------+-----------------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+---------------------+---------------------+--------------------------+---------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+------------